# Web Scraping with BeautifulSoup4

Beautiful Soup is a library that reads messy HTML and lets you **navigate and search it** like a normal Python object, instead of manually parsing text with string operations.

In this lab, we'll scrape a course schedule page for a Master's Degree in Applied Data Science & Artificial Intelligence program, and turn it into a clean pandas DataFrame we can filter and analyze.

### How HTML is structured

Think of an HTML page like a set of nested boxes. A big box (like `<body>`) can contain smaller boxes inside it (like `<table>`), and those can contain even smaller boxes (like `<tr>` and `<td>`). Every box is called a **tag**, and tags can also carry extra labels called **attributes** (like `id="session_table"`).

Beautiful Soup reads this whole structure and lets us:
- Grab a specific box by its tag name (e.g. "give me the table")
- Search for every box of a certain kind (e.g. "give me all the rows")
- Look inside a box to see what's nested inside it

This is much easier than trying to find things in HTML using plain text searching.

<img src="./images/BS4_html.jpg" align="left" style="width:600px;"/>

### Our dataset for today

We'll use a small mock course schedule page, `schedule.html`, for a Master's Degree in Data Science & Artificial Intelligence program. It contains:
- A title and a few announcement paragraphs
- A table listing 22 course sessions (SN, Date, Course, Instructor, Room Number, Day, Time)


### 1. Reading an HTML File and Extracting Its Contents Using Beautiful Soup


- In this exercise, we will do the **simplest** thing possible. 
- We will import the Beautiful Soup or bs4 library and then use it to **read an HTML document**. 
- Then, we will **examine the different kinds of objects** it returns. <br>

While doing the exercises for this topic, you should have the example HTML file (called test.html) open in a text editor so that you can check for the different tags and their attributes and contents:

00. **Install the bs4 library:**

In [77]:
#!pip install beautifulsoup4

**Import the bs4 library:**

In [78]:
from bs4 import BeautifulSoup

You can **pass a file handler directly to the constructor of the BeautifulSoup** object and it will read the contents from the file that the handler is attached to.  
We will see that the return type is an instance of bs4.BeautifulSoup.  
This class holds **all the methods we need to navigate through the DOM tree** that the document represents.

**Use bs4 to read the html file from the disk**

In [79]:
with open("datasets/schedule.html", "r") as fd:
    soup = BeautifulSoup(fd, "html.parser")

print(type(soup))

<class 'bs4.BeautifulSoup'>


**Note:** We explicitly passed `"html.parser"` when creating the `BeautifulSoup` object. If you don't specify a parser, newer versions of BS4 will show a warning (it still works, but it's best practice to always name the parser you want to use).

**soup.prettify()** takes the parsed tree and returns it back out as a nicely formatted string with:

- Consistent line breaks (each tag typically gets its own line)
- Indentation(mirroring the tree structure we saw in the DOM diagram earlier)

In [80]:
print(soup.prettify())

<!DOCTYPE html>
<html>
 <head>
  <title>
   Master's Degree in Data Science &amp; Artificial Intelligence: Course Schedule
  </title>
  <style>
   table {
            border-collapse: collapse;
            width: 100%;
        }
        th, td {
            border: 1px solid black;
            padding: 6px 10px;
            text-align: left;
        }
  </style>
 </head>
 <body>
  <h1>
   Master's Degree in Data Science &amp; Artificial Intelligence
  </h1>
  <p>
   Welcome to the
   <b>
    Master's Degree in Data Science &amp; Artificial Intelligence
   </b>
   program. Below you will find the current schedule of sessions for this term.
  </p>
  <p>
   All sessions are held on campus unless otherwise noted. Please check the room assignment, date, and day before each class, as schedules may change week to week.
  </p>
  <p>
   If you have any questions about the schedule, please contact the
   <b>
    program office
   </b>
   during regular business hours.
  </p>
  <h2>
   Instructor

### 2. Accessing a single tag

If you just want the *first* tag of a certain kind, you can access it directly using dot notation, like accessing a normal attribute of an object.

In [81]:
# Show only the first <p> tag
print(soup.p)

<p>Welcome to the <b>Master's Degree in Data Science &amp; Artificial Intelligence</b> program. Below you will find the current schedule of sessions for this term.</p>


| Dot notation | What it returns |
|---|---|
| `soup.title` | The `<title>` tag inside `<head>` |
| `soup.h1` | The main heading |
| `soup.h2` | The first `<h2>` |
| `soup.b` | The first bold tag|
| `soup.table` | The first table|
| `soup.tr` | The first row of the first table |
| `soup.th` | The first header cell|
| `soup.td` | The first data cell  |
| `soup.style` | The `<style>` block inside `<head>` |

### 3. Using find_all() to get every matching tag

`find_all()` searches the entire document and returns **every** tag that matches, as a list. This is the go-to method whenever you need more than just the first result.

In [82]:
all_ps = soup.find_all('p')
all_ps

[<p>Welcome to the <b>Master's Degree in Data Science &amp; Artificial Intelligence</b> program. Below you will find the current schedule of sessions for this term.</p>,
 <p>All sessions are held on campus unless otherwise noted. Please check the room assignment, date, and day before each class, as schedules may change week to week.</p>,
 <p>If you have any questions about the schedule, please contact the <b>program office</b> during regular business hours.</p>]

In [83]:
len(all_ps)

3

In [84]:
# Select a specific paragraph by index
all_ps[2]

<p>If you have any questions about the schedule, please contact the <b>program office</b> during regular business hours.</p>

If you don't want the raw tag markup in your output, just the readable text, use `.get_text()` (or `.text`).

In [85]:
# Just the text, no tags
print(soup.p.get_text())

Welcome to the Master's Degree in Data Science & Artificial Intelligence program. Below you will find the current schedule of sessions for this term.


### 4. Getting the table


Our page now has **two** tables, instructor info and the course schedule.

In [86]:
# There are now 2 tables on the page, find_all returns both

all_tables = soup.find_all('table')
print(len(all_tables))

2


`soup.table` would only give us the *first* table on the page (instructor info), which isn't what we want. To reliably get the schedule table, we can select it by its `id` attribute.

In [87]:
# Target the schedule table specifically by its id
schedule_table = soup.find('table', id='session_table')
print(schedule_table.prettify)

<bound method Tag.prettify of <table id="session_table">
<tr>
<th>SN</th>
<th>Date</th>
<th>Course</th>
<th>Instructor</th>
<th>Room Number</th>
<th>Day</th>
<th>Time</th>
</tr>
<tr>
<td>1</td>
<td>Aug 24, 2026</td>
<td>Python Programming Basics</td>
<td>Dr. Thomas Becker</td>
<td>Room 101</td>
<td>Monday</td>
<td>01:00 PM</td>
</tr>
<tr>
<td>2</td>
<td>Aug 25, 2026</td>
<td>Data Structures &amp; Algorithms</td>
<td>Prof. Daniel Kessler</td>
<td>Room 204</td>
<td>Tuesday</td>
<td>09:00 AM</td>
</tr>
<tr>
<td>3</td>
<td>Aug 28, 2026</td>
<td>Statistics for Data Science</td>
<td>Dr. Thomas Becker</td>
<td>Room 102</td>
<td>Friday</td>
<td>04:00 PM</td>
</tr>
<tr>
<td>4</td>
<td>Aug 31, 2026</td>
<td>NumPy &amp; Array Computing</td>
<td>Dr. Yuki Tanaka</td>
<td>Room 101</td>
<td>Monday</td>
<td>09:00 AM</td>
</tr>
<tr>
<td>5</td>
<td>Sep 1, 2026</td>
<td>Pandas for Data Analysis</td>
<td>Prof. Daniel Kessler</td>
<td>Room 101</td>
<td>Tuesday</td>
<td>04:00 PM</td>
</tr>
<tr>
<td>6</td>
<

=== Task 1 === (5 points)

1. Using dot notation, print the first `<h2>` tag on the page
   
2. Using dot notation, print the first `<table>` tag on the page
   
3. Using `find_all()`, get every `<p>` tag on the page and print how many there are
   
4. From that list, print just the **text** (no tags) of the last paragraph using `.get_text()`
   
5. Use `.find('table', id=...)` to select the `session_table` specifically and print its `.prettify()`

In [88]:
# 1. Using dot notation, print the first <h2> tag on the page

In [89]:
# 2. Using dot notation, print the first <table> tag on the page
# Which table does this return, and why?

In [90]:
# 3. Using find_all(), get every <p> tag on the page and print how many there are

In [91]:
# 4. From that list, print just the text (no tags) of the last paragraph using .get_text()

In [92]:
# 5. Use .find('table', id=...) to select the session_table specifically and print its .prettify()

### 5. Navigating the tree: children vs. descendants

We can also walk through a tag's contents directly, without searching by name.

- **`.children`** gives only the tag's *direct* children (one level down).
- **`.descendants`** gives *everything* nested inside it, at every level, children, grandchildren, and so on.


<img src="./images/children.png" align="center" style="width:1200px;"/>

In [93]:
schedule_table = soup.find('table', id='session_table')

children = list(schedule_table.children)
descendants = list(schedule_table.descendants)

print(f"Number of children: {len(children)}")
print(f"Number of descendants: {len(descendants)}")

Number of children: 47
Number of descendants: 553


In [94]:
first_p = soup.p

p_children = list(first_p.children)
p_descendants = list(first_p.descendants)

print(f"Number of children: {len(p_children)}")
print(f"Number of descendants: {len(p_descendants)}")

Number of children: 3
Number of descendants: 4


Notice `descendants` is a much bigger list than `children`, that's because it also counts every `<td>` and every bit of text inside each row, not just the rows themselves.

In [95]:
print()
print("--- First 10 children ---")
for i, c in enumerate(children[:10]):
    print(i, repr(c)[:70])

print()
print("--- First 10 descendants ---")
for i, d in enumerate(descendants[:10]):
    print(i, repr(d)[:70])


--- First 10 children ---
0 '\n'
1 <tr>
<th>SN</th>
<th>Date</th>
<th>Course</th>
<th>Instructor</th>
<th
2 '\n'
3 <tr>
<td>1</td>
<td>Aug 24, 2026</td>
<td>Python Programming Basics</t
4 '\n'
5 <tr>
<td>2</td>
<td>Aug 25, 2026</td>
<td>Data Structures &amp; Algori
6 '\n'
7 <tr>
<td>3</td>
<td>Aug 28, 2026</td>
<td>Statistics for Data Science<
8 '\n'
9 <tr>
<td>4</td>
<td>Aug 31, 2026</td>
<td>NumPy &amp; Array Computing<

--- First 10 descendants ---
0 '\n'
1 <tr>
<th>SN</th>
<th>Date</th>
<th>Course</th>
<th>Instructor</th>
<th
2 '\n'
3 <th>SN</th>
4 'SN'
5 '\n'
6 <th>Date</th>
7 'Date'
8 '\n'
9 <th>Course</th>


| Index | children | descendants |
|---|---|---|
| 0 | `'\n'` | `'\n'` |
| 1 | `<tr><th>SN</th><th>Date</th>...` | `<tr><th>SN</th><th>Date</th>...` |
| 2 | `'\n'` | `'\n'` |
| 3 | `<tr><td>1</td><td>Aug 24, 2026</td>...` | `<th>SN</th>` |
| 4 | `'\n'` | `'SN'` |
| 5 | `<tr><td>2</td><td>Aug 25, 2026</td>...` | `'\n'` |
| 6 | `'\n'` | `<th>Date</th>` |
| 7 | `<tr><td>3</td><td>Aug 28, 2026</td>...` | `'Date'` |
| 8 | `'\n'` | `'\n'` |
| 9 | `<tr><td>4</td><td>Aug 31, 2026</td>...` | `<th>Course</th>` |

In [96]:
# Index 1 is the header row (index 0 is just whitespace before it)
list(schedule_table.children)[1]

<tr>
<th>SN</th>
<th>Date</th>
<th>Course</th>
<th>Instructor</th>
<th>Room Number</th>
<th>Day</th>
<th>Time</th>
</tr>

In [97]:
# Index 3 is the first actual data row
list(schedule_table.children)[3]

<tr>
<td>1</td>
<td>Aug 24, 2026</td>
<td>Python Programming Basics</td>
<td>Dr. Thomas Becker</td>
<td>Room 101</td>
<td>Monday</td>
<td>01:00 PM</td>
</tr>

`.descendants` walks through everything nested inside a tag, one item at a time, tags **and** the text inside them. Let's look at the full list, then narrow in on a single row.

In [98]:
list(schedule_table.descendants)[:30]

['\n',
 <tr>
 <th>SN</th>
 <th>Date</th>
 <th>Course</th>
 <th>Instructor</th>
 <th>Room Number</th>
 <th>Day</th>
 <th>Time</th>
 </tr>,
 '\n',
 <th>SN</th>,
 'SN',
 '\n',
 <th>Date</th>,
 'Date',
 '\n',
 <th>Course</th>,
 'Course',
 '\n',
 <th>Instructor</th>,
 'Instructor',
 '\n',
 <th>Room Number</th>,
 'Room Number',
 '\n',
 <th>Day</th>,
 'Day',
 '\n',
 <th>Time</th>,
 'Time',
 '\n',
 '\n',
 <tr>
 <td>1</td>
 <td>Aug 24, 2026</td>
 <td>Python Programming Basics</td>
 <td>Dr. Thomas Becker</td>
 <td>Room 101</td>
 <td>Monday</td>
 <td>01:00 PM</td>
 </tr>,
 '\n',
 <td>1</td>,
 '1',
 '\n']

Each row (`<tr>`) is itself a tag with its own children and descendants. Let's zoom into just one row and see what's inside it.

In [99]:
# Row at index 5 in the table's children (the 2nd course row)
row = list(schedule_table.children)[5]
print(row)

<tr>
<td>2</td>
<td>Aug 25, 2026</td>
<td>Data Structures &amp; Algorithms</td>
<td>Prof. Daniel Kessler</td>
<td>Room 204</td>
<td>Tuesday</td>
<td>09:00 AM</td>
</tr>


In [100]:
# Look at everything nested inside this one row
list(row.descendants)

['\n',
 <td>2</td>,
 '2',
 '\n',
 <td>Aug 25, 2026</td>,
 'Aug 25, 2026',
 '\n',
 <td>Data Structures &amp; Algorithms</td>,
 'Data Structures & Algorithms',
 '\n',
 <td>Prof. Daniel Kessler</td>,
 'Prof. Daniel Kessler',
 '\n',
 <td>Room 204</td>,
 'Room 204',
 '\n',
 <td>Tuesday</td>,
 'Tuesday',
 '\n',
 <td>09:00 AM</td>,
 '09:00 AM',
 '\n']

Notice the pattern: each `<td>` tag is followed immediately by its text content as a separate item in the list. That's why `row.descendants` has both the tag *and* its text as separate entries.

=== Task 2 === (5 points)

1. Use `soup.find('table', id='instructor_table')` to get the instructor table, then print the number of `.children` and `.descendants` it has
   
2. Print the first 10 items of `.children` and the first 10 items of `.descendants` for the instructor table, with their index numbers
   
3. Using `.children`, extract the row for the **6th instructor** (Dr. Yuki Tanaka) and print it
   
4. Using `.descendants` on that same row, find the index number where the instructor's **email** (`yuki.tanaka@cpdsai.edu`) appears
   
5. Using indexing into `.descendants` (no `find_all`), print just the **office hour** for that same row

In [101]:
# 1. Get the instructor table, print number of .children and .descendants

In [102]:
# 2. Print first 10 items of .children and first 10 items of .descendants, with index numbers

In [103]:
# 3. Using .children, extract the row for the 6th instructor (Dr. Yuki Tanaka)

In [104]:
# 4. Using .descendants on that row, find the index number where the email appears

In [105]:
# 5. Using indexing into .descendants, print just the office hour for that row

### 7. From table to DataFrame

- Now, we are going to go one step further and use bs4 combined with pandas to generate a DataFrame out of a plain HTML table.
- We will extract the data from the test.html page using the BeautifulSoup library. 
- We will then perform a few operations for data preparation and display the data in an easily readable tabular format.

In [106]:
# Import Necessary libraries

#!pip install pandas
import pandas as pd
from bs4 import BeautifulSoup

In [107]:
rows = schedule_table.find_all('tr')
rows

[<tr>
 <th>SN</th>
 <th>Date</th>
 <th>Course</th>
 <th>Instructor</th>
 <th>Room Number</th>
 <th>Day</th>
 <th>Time</th>
 </tr>,
 <tr>
 <td>1</td>
 <td>Aug 24, 2026</td>
 <td>Python Programming Basics</td>
 <td>Dr. Thomas Becker</td>
 <td>Room 101</td>
 <td>Monday</td>
 <td>01:00 PM</td>
 </tr>,
 <tr>
 <td>2</td>
 <td>Aug 25, 2026</td>
 <td>Data Structures &amp; Algorithms</td>
 <td>Prof. Daniel Kessler</td>
 <td>Room 204</td>
 <td>Tuesday</td>
 <td>09:00 AM</td>
 </tr>,
 <tr>
 <td>3</td>
 <td>Aug 28, 2026</td>
 <td>Statistics for Data Science</td>
 <td>Dr. Thomas Becker</td>
 <td>Room 102</td>
 <td>Friday</td>
 <td>04:00 PM</td>
 </tr>,
 <tr>
 <td>4</td>
 <td>Aug 31, 2026</td>
 <td>NumPy &amp; Array Computing</td>
 <td>Dr. Yuki Tanaka</td>
 <td>Room 101</td>
 <td>Monday</td>
 <td>09:00 AM</td>
 </tr>,
 <tr>
 <td>5</td>
 <td>Sep 1, 2026</td>
 <td>Pandas for Data Analysis</td>
 <td>Prof. Daniel Kessler</td>
 <td>Room 101</td>
 <td>Tuesday</td>
 <td>04:00 PM</td>
 </tr>,
 <tr>
 <td>6</

In [108]:
headers = rows[0]
headers

<tr>
<th>SN</th>
<th>Date</th>
<th>Course</th>
<th>Instructor</th>
<th>Room Number</th>
<th>Day</th>
<th>Time</th>
</tr>

In [109]:
data_rows = rows[1:]

print(f"Number of data rows: {len(data_rows)}")

Number of data rows: 22


Next, pull the column names out of the header row using `find_all('th')`.

In [110]:
col_headers = []
for th in headers.find_all('th'):
    col_headers.append(th.get_text())

col_headers

['SN', 'Date', 'Course', 'Instructor', 'Room Number', 'Day', 'Time']

Now let's extract the actual data from each row. We need a **two-dimensional list** (a list of lists), one inner list per row, so pandas can turn it into a DataFrame.

In [111]:
# Build a 2D list: one row per course session
df_data = []
for tr in data_rows:
    row = []
    for td in tr.find_all('td'):
        row.append(td.get_text())
    df_data.append(row)

df_data[:3]

[['1',
  'Aug 24, 2026',
  'Python Programming Basics',
  'Dr. Thomas Becker',
  'Room 101',
  'Monday',
  '01:00 PM'],
 ['2',
  'Aug 25, 2026',
  'Data Structures & Algorithms',
  'Prof. Daniel Kessler',
  'Room 204',
  'Tuesday',
  '09:00 AM'],
 ['3',
  'Aug 28, 2026',
  'Statistics for Data Science',
  'Dr. Thomas Becker',
  'Room 102',
  'Friday',
  '04:00 PM']]

With both the headers and the data ready, we can now create the DataFrame.

In [112]:
schedule_df = pd.DataFrame(df_data, columns=col_headers)
schedule_df.head()

,SN,Date,Course,Instructor,Room Number,Day,Time
0,1,"Aug 24, 2026",Python Programming Basics,Dr. Thomas Becker,Room 101,Monday,01:00 PM
1,2,"Aug 25, 2026",Data Structures & Algorithms,Prof. Daniel Kessler,Room 204,Tuesday,09:00 AM
2,3,"Aug 28, 2026",Statistics for Data Science,Dr. Thomas Becker,Room 102,Friday,04:00 PM
3,4,"Aug 31, 2026",NumPy & Array Computing,Dr. Yuki Tanaka,Room 101,Monday,09:00 AM
4,5,"Sep 1, 2026",Pandas for Data Analysis,Prof. Daniel Kessler,Room 101,Tuesday,04:00 PM


### 8. Scraping data from a live website

We'll scrape a Wikipedia table of country populations and turn it into a DataFrame.

`requests` is a separate library (not built into Python) specifically for making HTTP requests, actually reaching out over the internet to fetch a page.

In [113]:
import requests
from bs4 import BeautifulSoup

url = 'https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_population_(United_Nations)'

The address of the live page we want to scrape. Before: `soup = BeautifulSoup(fd, ...)` opened a file on disk, but here we don't have a file, we need to actually download the page's HTML first.

In [114]:
# Wikipedia (like many sites) blocks requests that don't look like they're
# Define a custom User-Agent header to mimic a web browser

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
}

try:
    # Make the GET request with the custom headers
    r = requests.get(url, headers=headers, timeout=15)
    r.raise_for_status()  # raises an error for bad responses (404, 500, etc.)
    print(f"Success! Status code: {r.status_code}")
    
except requests.exceptions.RequestException as e:
    print(f"Request failed: {e}")

Success! Status code: 200


In [115]:
live_soup = BeautifulSoup(r.text, "html.parser")

# Wikipedia data tables use the class "wikitable"

all_tables = live_soup.find_all('table', {'class': 'wikitable'})

print(f"Number of wikitables found: {len(all_tables)}")

Number of wikitables found: 1


Just like our schedule page had multiple tables, this Wikipedia article may have more than one `wikitable`. Let's inspect the first one, since that's usually the main population table.

In [116]:
population_table = all_tables[0]
rows = population_table.find_all('tr')
print(f"Number of rows: {len(rows)}")

Number of rows: 240


In [117]:
headers_row = rows[0]
col_headers = [th.get_text(strip=True) for th in headers_row.find_all('th')]
col_headers

['Country or territory',
 'Population(1 July 2022)',
 'Population(1 July 2023)',
 'Change(%)',
 'UN continentalregion[1]',
 'UN statisticalsubregion[1]']

Real-world scraped data is rarely clean, this table includes footnote markers (like `[a]`) and extra whitespace in some cells. We'll do a small cleanup pass after extracting it.

In [120]:
import re

data_rows = rows[1:]

def clean_cell(text):
    # Remove footnote markers like [a] or [1], and strip extra whitespace
    return re.sub(r'\[.*?\]', '', text).strip()

df_data = [[clean_cell(td.get_text()) for td in tr.find_all(['th', 'td'])] for tr in data_rows]
df_data[:3]

[[],
 ['World', '8,021,407,192', '8,091,734,930', '+0.88%', '–', '–'],
 ['India',
  '1,425,423,212',
  '1,438,069,596',
  '+0.89%',
  'Asia',
  'Southern Asia']]

In [121]:
population_df = pd.DataFrame(df_data, columns=col_headers)
population_df.head()

,Country or territory,Population(1 July 2022),Population(1 July 2023),Change(%),UN continentalregion[1],UN statisticalsubregion[1]
0,None,None,None,None,None,None
1,World,"8,021,407,192","8,091,734,930",+0.88%,–,–
2,India,"1,425,423,212","1,438,069,596",+0.89%,Asia,Southern Asia
3,China,"1,425,179,569","1,422,584,933",−0.18%,Asia,Eastern Asia
4,United States,"341,534,046","343,477,335",+0.57%,Americas,Northern America
